# 20.3 Practical considerations in integer programming

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

As a rule, any `integer` program is much harder to `solve` than a linear program of the
same size and structure. A rough idea of the difference in the previous examples is given
by the number of iterations reported by the solvers; it is 41 for solving the original linear
multicommodity transportation problem, but ranges from about 280 to 400 for the mixed
`integer` versions. The computation times vary similarly; the linear programs are solved
almost instantly, while the mixed `integer` ones are noticeably slower.

As the size of the problem increases, the difficulty of an `integer` program also grows
more quickly than the difficulty of a comparable linear program. Thus the practical limits
to the size of a solvable `integer` program will be much more restrictive. Indeed, AMPL
can easily generate `integer` programs that are too difficult for your computer to `solve` in a
reasonable amount of time or memory. Before you make a commitment to use a `model`
with `integer` variables, therefore, you should consider whether an alternative continuous
linear or network formulation might give adequate answers. If you must use an `integer`
formulation, try it on collections of `data` that increase gradually in size, so that you can
get an idea of the computer resources required.

If you do encounter an `integer` program that is difficult to `solve`, you should consider
reformulations that can make it easier. An `integer` programming solver attempts to investigate
all of the different possible combinations of values of the `integer` variables;
although it employs a sophisticated strategy that rules out the vast majority of combinations
as infeasible or suboptimal, the number of combinations remaining to be checked
can still be huge. Thus you should try to use as few `integer` variables as possible. For
those variables that are not zero-one, lower and upper bounds should be made as tight as
possible to reduce the number of combinations that might be investigated.

Solvers derive valuable information about the solutions to an `integer` program by fixing
or restricting some of the `integer` variables, then solving the linear programming
"relaxation" that results when the remaining `integer` restrictions are dropped. You may
be able to assist in this strategy by reformulating the `model` so that the solution of a relaxation
will be closer to the solution of the associated `integer` problem. In the multicommodity
transportation `model`, for example, if Use[i,j] is 0 then each of the individual
variables Trans[i,j,p] must be 0, while if Use[i,j] is 1 then Trans[i,j,p]
cannot be larger than either supply[i,p] or demand[j,p]. This suggests that we
add the following constraints:

```ampl
subject to Avail {i in ORIG, j in DEST, p in PROD}:
          Trans[i,j,p] <= min(supply[i,p],demand[j,p]) * Use[i,j];
```

Although these constraints do not rule out any previously admissible `integer` solutions,
they tend to force Use[i,j] to be closer to 1 for any solution of the relaxation that uses
the route from `i` to j. As a result, the relaxation is more accurate, and may help the
solver to find the optimum `integer` solution more quickly; this advantage may outweigh
the extra cost of handling more constraints. Tradeoffs of this kind are most often
observed for problems substantially larger than the examples in this section, however.

As this example suggests, the choice of a formulation is much more critical in `integer`
than in linear programming. For large problems, solution times can change dramatically
in response to simple reformulations. The effect of a reformulation can be hard to predict,
however; it depends on the structure of your `model` and `data`, and on the details of
the strategy used by your solver. Generally you will need to do some experimentation to
see what works best.

You can also try to help a solver by changing some of the default settings that determine
how it initially processes a problem and how it searches for `integer` solutions. Many
of these settings can be manipulated from within AMPL, as explained in the separate
instructions for using particular solvers.

Finally, a solver may provide options for stopping prematurely, returning an `integer`
solution that has been determined to bring the `objective` value to within a small percentage
of optimality. In some cases, such a solution is found relatively early in the solution
process; you may save a great deal of computation time by not insisting that the solver go
on to find a provably optimal `integer` solution.

In summary, experimentation is usually necessary to `solve` `integer` or mixed-`integer`
linear programs. Your chances of success are greatest if you approach the use of `integer`
variables with caution, and if you are willing to keep trying alternative formulations, settings
and strategies until you get an acceptable result.

**Bibliography**

Ellis L. Johnson, Michael M. Kostreva and Uwe H. Suhl, "Solving 0-1 Integer Programming Problems
Arising from Large-Scale Planning Models." Operations Research 33 (1985) pp. 803-819.

A case study in which preprocessing, reformulation, and algorithmic strategies were brought to
bear on the solution of a difficult class of `integer` linear programs.
George L. Nemhauser and Laurence A. Wolsey, Integer and Combinatorial Optimization. John
Wiley & Sons (New York, NY, 1988). A survey of `integer` programming problems, theory and
algorithms.

Alexander Schrijver, Theory of Linear and Integer Programming. John Wiley & Sons (New York,
NY, 1986). A guide to the fundamentals of the subject, with a particularly thorough collection of
references.

Laurence A. Wolsey, Integer Programming. Wiley-Interscience (New York, NY, 1998). A practical,
intermediate-level guide for readers familiar with linear programming.